# Animal Face Classification

In [ ]:
# import kagglehub
# import os
# import shutil

# target_dir = os.path.expanduser("~/Desktop/LEARNINGPANDAS/10_Pytorch/data/animal_faces")

# download_path = kagglehub.dataset_download("andrewmvd/animal-faces")

# os.makedirs(os.path.dirname(target_dir), exist_ok=True)

# if os.path.exists(target_dir):
#     shutil.rmtree(target_dir)

# shutil.move(download_path, target_dir)

# print(f"Dataset successfully moved to: {target_dir}")

In [1]:
!pip install torchsummary

In [2]:
import torch
import torch.nn as nn 
import torch.optim as optim
from torchvision.transforms import transforms
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from PIL import Image
import pandas as pd
import numpy as np
import os

In [3]:
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')

In [4]:
image_path = []
labels = []

for i in os.listdir('/Users/atharvakhandelwal/Desktop/learningpandas/10_Pytorch/data/animal_faces'):
    for label in os. listdir(f"/Users/atharvakhandelwal/Desktop/learningpandas/10_Pytorch/data/animal_faces/{i}"):
        for image in os. listdir(f"/Users/atharvakhandelwal/Desktop/learningpandas/10_Pytorch/data/animal_faces/{i}/{label}"):
            image_path.append(f"/Users/atharvakhandelwal/Desktop/learningpandas/10_Pytorch/data/animal_faces/{i}/{label}/{image}")
            labels.append(label)


df = pd.DataFrame(zip(image_path, labels), columns = ["image_path", "labels"])

In [5]:
df.sample(5)

,image_path,labels
15669,/Users/atharvakhandelwal/Desktop/learningpanda...,wild
6663,/Users/atharvakhandelwal/Desktop/learningpanda...,dog
3235,/Users/atharvakhandelwal/Desktop/learningpanda...,cat
12037,/Users/atharvakhandelwal/Desktop/learningpanda...,wild
4535,/Users/atharvakhandelwal/Desktop/learningpanda...,cat


In [6]:
train = df.sample(frac=0.7)
test = df.drop(train.index)
val = test.sample(frac=0.5)
test = test.drop(val.index)

In [7]:
encoder = LabelEncoder()
encoder.fit(df['labels'])

transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


In [8]:
class CustomImageDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform
        self.labels = encoder.transform(self.dataframe['labels'])

    def __len__(self):
        return self.dataframe.shape[0]
    
    def __getitem__(self, index):
        img_path = self.dataframe.iloc[index, 0]
        label = self.labels[index]

        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.long)


In [9]:
train_dataset = CustomImageDataset(train, transform)
val_dataset = CustomImageDataset(val, transform)
test_dataset = CustomImageDataset(test, transform)

In [10]:
learning_rate = 1e-4
batch_size = 16
epochs = 10

In [11]:
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=True)

In [ ]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()

        self.Conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.Conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.Conv3 = nn.Conv2d(64,128, kernel_size=3, padding=1)
        self.Conv4 = nn.Conv2d(128,256, kernel_size=3, padding=1)

        self.pooling = nn.MaxPool2d(2,2)

        self.relu = nn.ReLU()

        self.flatten = nn.Flatten()
        self.linear = nn.Linear((256*8*8), 128)

        self.output = nn.Linear(128, len(df['labels'].unique()))

    def forward(self, x):
        x = self.Conv1(x)
        x = self.pooling(x)
        x = self.relu(x)

        x = self.Conv2(x)
        x = self.pooling(x)
        x = self.relu(x)

        x = self.Conv3(x)
        x = self.pooling(x)
        x = self.relu(x)

        x = self.Conv4(x)
        x = self.pooling(x)
        x = self.relu(x)

        x = self.flatten(x)

        x = self.linear(x)
        x = self.output(x)

        return x

In [12]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),  # (B, 256, 1, 1)
            nn.Flatten(),
            nn.Linear(256, 3)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


In [13]:
model = Net().to(device)

In [14]:
from torchinfo import summary

summary(model, input_size=(1, 3, 128, 128), device="mps")

Layer (type:depth-idx)                   Output Shape              Param #
Net                                      [1, 3]                    --
├─Sequential: 1-1                        [1, 256, 8, 8]            --
│    └─Conv2d: 2-1                       [1, 32, 128, 128]         896
│    └─BatchNorm2d: 2-2                  [1, 32, 128, 128]         64
│    └─ReLU: 2-3                         [1, 32, 128, 128]         --
│    └─MaxPool2d: 2-4                    [1, 32, 64, 64]           --
│    └─Conv2d: 2-5                       [1, 64, 64, 64]           18,496
│    └─BatchNorm2d: 2-6                  [1, 64, 64, 64]           128
│    └─ReLU: 2-7                         [1, 64, 64, 64]           --
│    └─MaxPool2d: 2-8                    [1, 64, 32, 32]           --
│    └─Conv2d: 2-9                       [1, 128, 32, 32]          73,856
│    └─BatchNorm2d: 2-10                 [1, 128, 32, 32]          256
│    └─ReLU: 2-11                        [1, 128, 32, 32]          --
│   

In [15]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

In [16]:
from tqdm import tqdm

for epoch in range(epochs):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    progress_bar = tqdm(
        train_loader,
        desc=f"Epoch [{epoch+1}/{epochs}]",
        leave=True
    )

    for inputs, labels in progress_bar:
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        # metrics
        batch_size = inputs.size(0)
        running_loss += loss.item() * batch_size
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += batch_size

        # update tqdm display
        progress_bar.set_postfix({
            "loss": f"{running_loss / total:.4f}",
            "acc": f"{correct / total:.4f}"
        })

    epoch_loss = running_loss / total
    epoch_acc = correct / total

    print(f"Epoch [{epoch+1}/{epochs}] "
          f"Loss: {epoch_loss:.4f} "
          f"Acc: {epoch_acc:.4f}")


Epoch [1/10]: 100%|██████████| 706/706 [01:01<00:00, 11.56it/s, loss=0.6918, acc=0.6988]


Epoch [1/10] Loss: 0.6918 Acc: 0.6988


Epoch [2/10]: 100%|██████████| 706/706 [01:00<00:00, 11.59it/s, loss=0.4819, acc=0.8130]


Epoch [2/10] Loss: 0.4819 Acc: 0.8130


Epoch [3/10]: 100%|██████████| 706/706 [01:02<00:00, 11.37it/s, loss=0.3877, acc=0.8581]


...

Epoch [8/10] Loss: 0.1873 Acc: 0.9356


Epoch [9/10]: 100%|██████████| 706/706 [01:04<00:00, 11.00it/s, loss=0.1625, acc=0.9445]


Epoch [9/10] Loss: 0.1625 Acc: 0.9445


Epoch [10/10]: 100%|██████████| 706/706 [01:01<00:00, 11.49it/s, loss=0.1442, acc=0.9524]

Epoch [10/10] Loss: 0.1442 Acc: 0.9524


In [17]:
model.eval()
val_loss = 0.0
val_correct = 0
val_total = 0

progress_bar = tqdm(val_loader, desc="Validation")

with torch.no_grad():
    for inputs, labels in progress_bar:
        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = model(inputs)
        loss = criterion(outputs, labels)

        batch_size = inputs.size(0)
        val_loss += loss.item() * batch_size
        _, preds = torch.max(outputs, 1)
        val_correct += (preds == labels).sum().item()
        val_total += batch_size

        progress_bar.set_postfix({
            "loss": f"{val_loss / val_total:.4f}",
            "acc": f"{val_correct / val_total:.4f}"
        })

val_loss /= val_total
val_acc = val_correct / val_total


Validation: 100%|██████████| 152/152 [00:09<00:00, 15.78it/s, loss=0.1287, acc=0.9599]


In [18]:
model.eval()
test_loss = 0.0
test_correct = 0
test_total = 0

progress_bar = tqdm(test_loader, desc="Test")

with torch.no_grad():
    for inputs, labels in progress_bar:
        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = model(inputs)
        loss = criterion(outputs, labels)

        batch_size = inputs.size(0)
        test_loss += loss.item() * batch_size
        _, preds = torch.max(outputs, 1)
        test_correct += (preds == labels).sum().item()
        test_total += batch_size

        progress_bar.set_postfix({
            "loss": f"{test_loss / test_total:.4f}",
            "acc": f"{test_correct / test_total:.4f}"
        })

test_loss /= test_total
test_acc = test_correct / test_total


Test: 100%|██████████| 152/152 [00:09<00:00, 16.44it/s, loss=0.1440, acc=0.9566]
